In [1]:
from matstruct_lab.heterostructures import build_heterostructure
from ase.io import read, write
from matstruct_lab.lattice_match import find_2d_matches, print_matches
from matstruct_lab.heterostructures import build_heterostructure

In [2]:
from pathlib import Path
import json

import pandas as pd
from ase.io import read, write

from matstruct_lab.heterostructures import build_electrode_sandwich
from matstruct_lab.sandwich_match import (
    find_2d_sandwich_matches,
    print_sandwich_matches,
    sandwich_layer_strain_records,
    write_csv_records,
    write_json_records,
)


# -----------------------------
# Inputs
# -----------------------------

electrode_path = "../examples/structures/Au_111.cif"
middle_path = "../examples/structures/MoS2.cif"

electrode_name = "Au111"
middle_name = "MoS2"

outdir = Path("sandwich_outputs")
outdir.mkdir(parents=True, exist_ok=True)

hard_max_atoms = 200

electrode_middle_distance = 3.3
vacuum = 20.0
shared_middle_weight = 0.5


# -----------------------------
# Read structures
# -----------------------------

electrode = read(electrode_path)
middle = read(middle_path)

print("Electrode atoms in input cell:", len(electrode))
print("Middle atoms in input cell:", len(middle))
print("Electrode cell:")
print(electrode.cell)
print("Middle cell:")
print(middle.cell)


# -----------------------------
# Cell sanity checks
# -----------------------------
# matstruct_lab assumes the first two lattice vectors are in-plane.

def check_2d_cell(atoms, name, tol=1e-6):
    cell = atoms.cell.array

    if abs(cell[0, 2]) > tol or abs(cell[1, 2]) > tol:
        raise ValueError(
            f"{name}: first two lattice vectors have nonzero z components. "
            "Reorient the slab so a,b are in-plane and c is out-of-plane."
        )

    if abs(cell[2, 0]) > tol or abs(cell[2, 1]) > tol:
        print(
            f"Warning: {name}: c vector has x/y components. "
            "An orthogonal vacuum direction is safer."
        )


check_2d_cell(electrode, electrode_name)
check_2d_cell(middle, middle_name)


# -----------------------------
# Sandwich-aware adaptive search
# -----------------------------
# max_atoms now counts:
#   bottom electrode + middle + top electrode
#
# max_strain is a fraction:
#   0.03 = 3%, 0.05 = 5%

search_attempts = [
    dict(max_entry=2, max_area=10, max_strain=0.030, max_atoms=80,  limit=10),
    dict(max_entry=3, max_area=16, max_strain=0.035, max_atoms=120, limit=15),
    dict(max_entry=4, max_area=25, max_strain=0.050, max_atoms=160, limit=20),
    dict(max_entry=5, max_area=40, max_strain=0.060, max_atoms=200, limit=30),
]

sandwich_matches = []
used_settings = None

for settings in search_attempts:
    settings = settings.copy()
    settings["max_atoms"] = min(settings["max_atoms"], hard_max_atoms)

    trial = find_2d_sandwich_matches(
        electrode=electrode,
        middle=middle,
        search_limit=1000,
        sort_by="atoms_then_strain",
        **settings,
    )

    if trial:
        sandwich_matches = trial
        used_settings = settings
        break

if not sandwich_matches:
    raise RuntimeError(
        "No sandwich matches found under 200 atoms. "
        "Try increasing max_strain slightly, max_area, or max_entry."
    )

print("Used search settings:")
print(used_settings)
print_sandwich_matches(sandwich_matches)


# -----------------------------
# Select match
# -----------------------------
# Since these are for computation, choose smallest final sandwich first,
# then lower strain.

selected = sandwich_matches[0]
match = selected.match

print("Selected sandwich match:")
print("sandwich atoms:", selected.sandwich_atoms)
print(f"max strain:  {100 * match.max_strain:.3f}%")
print(f"rms strain:  {100 * match.rms_strain:.3f}%")
print(f"area strain: {100 * match.area_strain:.3f}%")
print("electrode matrix:")
print(match.bottom_matrix)
print("middle matrix:")
print(match.top_matrix)


# -----------------------------
# Save candidate table
# -----------------------------

candidate_records = [sm.as_dict() for sm in sandwich_matches]

candidate_df = pd.DataFrame(candidate_records)
display(candidate_df)

write_csv_records(outdir / "sandwich_match_candidates_under_200_atoms.csv", candidate_records)
write_json_records(outdir / "sandwich_match_candidates_under_200_atoms.json", candidate_records)


# -----------------------------
# Build sandwich heterostructures
# -----------------------------

# Case 1: middle strained to electrodes.
# Electrodes stay fixed.
middle_strained = build_electrode_sandwich(
    electrode=electrode,
    middle=middle,
    match=match,
    electrode_middle_distance=electrode_middle_distance,
    vacuum=vacuum,
    strain="middle",
)

# Case 2: electrodes strained to middle.
# Middle stays fixed.
electrodes_strained = build_electrode_sandwich(
    electrode=electrode,
    middle=middle,
    match=match,
    electrode_middle_distance=electrode_middle_distance,
    vacuum=vacuum,
    strain="electrodes",
)

# Case 3: shared strain.
# Electrodes and middle are strained to an intermediate in-plane cell.
shared = build_electrode_sandwich(
    electrode=electrode,
    middle=middle,
    match=match,
    electrode_middle_distance=electrode_middle_distance,
    vacuum=vacuum,
    strain="shared",
    shared_middle_weight=shared_middle_weight,
)


# -----------------------------
# Atom-count checks and writing
# -----------------------------

structures = {
    f"{electrode_name}_{middle_name}_{electrode_name}_middle_strained": middle_strained,
    f"{electrode_name}_{middle_name}_{electrode_name}_electrodes_strained": electrodes_strained,
    f"{electrode_name}_{middle_name}_{electrode_name}_shared_strain": shared,
}

for name, atoms in structures.items():
    print(f"{name}: {len(atoms)} atoms")

    if len(atoms) > hard_max_atoms:
        raise RuntimeError(
            f"{name} has {len(atoms)} atoms, exceeding {hard_max_atoms}."
        )

    write(outdir / f"{name}.cif", atoms)
    write(outdir / f"{name}.xyz", atoms, format="extxyz")
    write(outdir / f"{name}.vasp", atoms, format="vasp", direct=True)

print("Wrote sandwich structures to:", outdir.resolve())


# -----------------------------
# Calculate actual layer strains
# -----------------------------

strain_records = sandwich_layer_strain_records(
    electrode=electrode,
    middle=middle,
    sandwich_match=selected,
    electrode_name=electrode_name,
    middle_name=middle_name,
    shared_middle_weight=shared_middle_weight,
)

strain_df = pd.DataFrame(strain_records)
display(strain_df)

write_csv_records(outdir / "selected_sandwich_actual_layer_strains.csv", strain_records)
write_json_records(outdir / "selected_sandwich_actual_layer_strains.json", strain_records)


# -----------------------------
# Metadata
# -----------------------------

metadata = {
    "electrode_path": electrode_path,
    "middle_path": middle_path,
    "electrode_name": electrode_name,
    "middle_name": middle_name,
    "hard_max_atoms": hard_max_atoms,
    "used_search_settings": used_settings,
    "selected_sandwich_match": selected.as_dict(),
    "electrode_middle_distance_A": electrode_middle_distance,
    "vacuum_A": vacuum,
    "shared_middle_weight": shared_middle_weight,
    "generated_structures": {
        name: {
            "num_atoms": len(atoms),
            "cif": str(outdir / f"{name}.cif"),
            "xyz": str(outdir / f"{name}.xyz"),
            "vasp": str(outdir / f"{name}.vasp"),
        }
        for name, atoms in structures.items()
    },
}

with open(outdir / "selected_sandwich_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print("Saved key files:")
print(outdir / "sandwich_match_candidates_under_200_atoms.csv")
print(outdir / "selected_sandwich_actual_layer_strains.csv")
print(outdir / "selected_sandwich_metadata.json")

Electrode atoms in input cell: 4
Middle atoms in input cell: 3
Electrode cell:
Cell([[2.883581453678741, 0.0, 0.0], [1.4417907268393704, 2.4972547927674507, 0.0], [0.0, 0.0, 37.06330319326588]])
Middle cell:
Cell([[3.18, 0.0, 0.0], [-1.5900000000000007, 2.7539607840345144, 0.0], [0.0, 0.0, 33.19]])
Used search settings:
{'max_entry': 3, 'max_area': 16, 'max_strain': 0.035, 'max_atoms': 120, 'limit': 15}
[0] original match index: 0
  sandwich atoms:          93
  electrode atoms total:   72
  middle atoms total:      21
  electrode matrix:
[[-3  0]
 [ 3 -3]]
  middle matrix:
[[-3 -2]
 [ 2 -1]]
  electrode area mult:     9
  middle area mult:        7
  max strain:              2.820%
  rms strain:              2.820%
  area strain:             5.719%

[1] original match index: 1
  sandwich atoms:          93
  electrode atoms total:   72
  middle atoms total:      21
  electrode matrix:
[[-3  0]
 [ 3 -3]]
  middle matrix:
[[ 3  2]
 [-2  1]]
  electrode area mult:     9
  middle area mul

,bottom_matrix,top_matrix,bottom_area_multiplier,top_area_multiplier,max_strain_percent,rms_strain_percent,area_strain_percent,total_atoms,match_index,sandwich_atoms,electrode_atoms_total,middle_atoms_total,electrode_matrix,middle_matrix,electrode_area_multiplier,middle_area_multiplier
0,"[[-3, 0], [3, -3]]","[[-3, -2], [2, -1]]",9,7,2.819938,2.819938,5.719397,57,0,93,72,21,"[[-3, 0], [3, -3]]","[[-3, -2], [2, -1]]",9,7
1,"[[-3, 0], [3, -3]]","[[3, 2], [-2, 1]]",9,7,2.819938,2.819938,5.719397,57,1,93,72,21,"[[-3, 0], [3, -3]]","[[3, 2], [-2, 1]]",9,7
2,"[[-3, 3], [-3, 0]]","[[-2, 1], [-3, -2]]",9,7,2.819938,2.819938,5.719397,57,2,93,72,21,"[[-3, 3], [-3, 0]]","[[-2, 1], [-3, -2]]",9,7
3,"[[-3, 3], [-3, 0]]","[[2, -1], [3, 2]]",9,7,2.819938,2.819938,5.719397,57,3,93,72,21,"[[-3, 3], [-3, 0]]","[[2, -1], [3, 2]]",9,7
4,"[[3, -3], [3, 0]]","[[-2, 1], [-3, -2]]",9,7,2.819938,2.819938,5.719397,57,4,93,72,21,"[[3, -3], [3, 0]]","[[-2, 1], [-3, -2]]",9,7
5,"[[3, -3], [3, 0]]","[[2, -1], [3, 2]]",9,7,2.819938,2.819938,5.719397,57,5,93,72,21,"[[3, -3], [3, 0]]","[[2, -1], [3, 2]]",9,7
6,"[[3, 0], [-3, 3]]","[[-3, -2], [2, -1]]",9,7,2.819938,2.819938,5.719397,57,6,93,72,21,"[[3, 0], [-3, 3]]","[[-3, -2], [2, -1]]",9,7
7,"[[3, 0], [-3, 3]]","[[3, 2], [-2, 1]]",9,7,2.819938,2.819938,5.719397,57,7,93,72,21,"[[3, 0], [-3, 3]]","[[3, 2], [-2, 1]]",9,7
8,"[[-3, 0], [0, -3]]","[[-1, -3], [2, -1]]",9,7,2.819938,2.819938,5.719397,57,8,93,72,21,"[[-3, 0], [0, -3]]","[[-1, -3], [2, -1]]",9,7
9,"[[-3, 0], [0, -3]]","[[1, 3], [-2, 1]]",9,7,2.819938,2.819938,5.719397,57,9,93,72,21,"[[-3, 0], [0, -3]]","[[1, 3], [-2, 1]]",9,7


Au111_MoS2_Au111_middle_strained: 93 atoms
Au111_MoS2_Au111_electrodes_strained: 93 atoms
Au111_MoS2_Au111_shared_strain: 93 atoms
Wrote sandwich structures to: /home/brian/Research/matstruct-lab/notebooks/sandwich_outputs


,reference,target,a_ref_A,b_ref_A,gamma_ref_deg,area_ref_A2,a_target_A,b_target_A,gamma_target_deg,area_target_A2,...,max_abs_principal_strain_percent,deformation_gradient,structure,strain_mode,layer,strained,electrode_matrix,middle_matrix,electrode_area_multiplier,middle_area_multiplier
0,electrode matched supercell,electrode matched supercell,8.650744,8.650744,120.0,64.809338,8.650744,8.650744,120.0,64.809338,...,0.000000e+00,"[[1.0, -0.0], [-0.0, 1.0]]",Au111_MoS2_Au111_middle_strained,middle,bottom_Au111,False,"[[-3, 0], [3, -3]]","[[-3, -2], [2, -1]]",9,7
1,middle matched supercell,electrode matched supercell,8.413489,8.413489,120.0,61.303167,8.650744,8.650744,120.0,64.809338,...,2.819938e+00,"[[0.7772456748460219, -0.6731144993982346], [0...",Au111_MoS2_Au111_middle_strained,middle,MoS2,True,"[[-3, 0], [3, -3]]","[[-3, -2], [2, -1]]",9,7
2,electrode matched supercell,electrode matched supercell,8.650744,8.650744,120.0,64.809338,8.650744,8.650744,120.0,64.809338,...,0.000000e+00,"[[1.0, -0.0], [-0.0, 1.0]]",Au111_MoS2_Au111_middle_strained,middle,top_Au111,False,"[[-3, 0], [3, -3]]","[[-3, -2], [2, -1]]",9,7
3,electrode matched supercell,middle matched supercell,8.650744,8.650744,120.0,64.809338,8.413489,8.413489,120.0,61.303167,...,2.742599e+00,"[[0.7351968494926338, 0.6366991484429052], [-0...",Au111_MoS2_Au111_electrodes_strained,electrodes,bottom_Au111,True,"[[-3, 0], [3, -3]]","[[-3, -2], [2, -1]]",9,7
4,middle matched supercell,middle matched supercell,8.413489,8.413489,120.0,61.303167,8.413489,8.413489,120.0,61.303167,...,1.110223e-14,"[[0.9999999999999999, 0.0], [1.151819518648436...",Au111_MoS2_Au111_electrodes_strained,electrodes,MoS2,False,"[[-3, 0], [3, -3]]","[[-3, -2], [2, -1]]",9,7
5,electrode matched supercell,middle matched supercell,8.650744,8.650744,120.0,64.809338,8.413489,8.413489,120.0,61.303167,...,2.742599e+00,"[[0.7351968494926338, 0.6366991484429052], [-0...",Au111_MoS2_Au111_electrodes_strained,electrodes,top_Au111,True,"[[-3, 0], [3, -3]]","[[-3, -2], [2, -1]]",9,7
6,electrode matched supercell,shared intermediate cell,8.650744,8.650744,120.0,64.809338,7.994680,7.994680,120.0,55.351937,...,7.583904e+00,"[[0.8675984247463169, 0.3183495742214526], [-0...",Au111_MoS2_Au111_shared_strain,shared,bottom_Au111,True,"[[-3, 0], [3, -3]]","[[-3, -2], [2, -1]]",9,7
7,middle matched supercell,shared intermediate cell,8.413489,8.413489,120.0,61.303167,7.994680,7.994680,120.0,55.351937,...,4.977827e+00,"[[0.888622837423011, -0.33655724969911727], [0...",Au111_MoS2_Au111_shared_strain,shared,MoS2,True,"[[-3, 0], [3, -3]]","[[-3, -2], [2, -1]]",9,7
8,electrode matched supercell,shared intermediate cell,8.650744,8.650744,120.0,64.809338,7.994680,7.994680,120.0,55.351937,...,7.583904e+00,"[[0.8675984247463169, 0.3183495742214526], [-0...",Au111_MoS2_Au111_shared_strain,shared,top_Au111,True,"[[-3, 0], [3, -3]]","[[-3, -2], [2, -1]]",9,7


Saved key files:
sandwich_outputs/sandwich_match_candidates_under_200_atoms.csv
sandwich_outputs/selected_sandwich_actual_layer_strains.csv
sandwich_outputs/selected_sandwich_metadata.json


In [5]:
from pathlib import Path
import json
import time

import pandas as pd
from ase.io import read, write

from matstruct_lab.heterostructures import build_electrode_sandwich
from matstruct_lab.sandwich_match import (
    find_2d_sandwich_matches,
    sandwich_layer_strain_records,
    write_csv_records,
    write_json_records,
)


# -----------------------------
# Inputs
# -----------------------------

electrode_path = "../examples/structures/Au_111.cif"
middle_path = "../examples/structures/MoS2.cif"

electrode_name = "Au111"
middle_name = "MoS2"

outdir = Path("small_sandwich_all_strain_modes")
outdir.mkdir(parents=True, exist_ok=True)

hard_max_atoms = 200

# Target actual strain on the strained material(s).
min_layer_strain_percent = 1.2
max_layer_strain_percent = 4.0

electrode_middle_distance = 3.3
vacuum = 20.0

# For shared mode only.
shared_middle_weight = 0.5

# Number of structures to write per mode.
n_to_write_per_mode = 5


# -----------------------------
# Read structures
# -----------------------------

electrode = read(electrode_path)
middle = read(middle_path)

print("Electrode atoms in input cell:", len(electrode))
print("Middle atoms in input cell:", len(middle))
print("Electrode cell:")
print(electrode.cell)
print("Middle cell:")
print(middle.cell)


# -----------------------------
# Cell sanity check
# -----------------------------

def check_2d_cell(atoms, name, tol=1e-6):
    cell = atoms.cell.array

    if abs(cell[0, 2]) > tol or abs(cell[1, 2]) > tol:
        raise ValueError(
            f"{name}: first two lattice vectors have nonzero z components. "
            "Reorient the slab so a,b are in-plane and c is out-of-plane."
        )

    if abs(cell[2, 0]) > tol or abs(cell[2, 1]) > tol:
        print(
            f"Warning: {name}: c vector has x/y components. "
            "An orthogonal vacuum direction is safer."
        )


check_2d_cell(electrode, electrode_name)
check_2d_cell(middle, middle_name)


# -----------------------------
# Strain extraction helpers
# -----------------------------

def mode_strain_summary(sm, mode):
    """
    Return Au and MoS2 actual strain for one sandwich match and one strain mode.

    mode = "middle", "electrodes", or "shared"
    """
    records = sandwich_layer_strain_records(
        electrode=electrode,
        middle=middle,
        sandwich_match=sm,
        electrode_name=electrode_name,
        middle_name=middle_name,
        shared_middle_weight=shared_middle_weight,
    )

    mode_records = [r for r in records if r["strain_mode"] == mode]

    electrode_rows = [
        r for r in mode_records
        if r["layer"] in [f"bottom_{electrode_name}", f"top_{electrode_name}"]
    ]

    middle_rows = [
        r for r in mode_records
        if r["layer"] == middle_name
    ]

    if not electrode_rows or not middle_rows:
        return None

    electrode_abs_strain = max(
        abs(r["max_abs_principal_strain_percent"])
        for r in electrode_rows
    )

    middle_abs_strain = max(
        abs(r["max_abs_principal_strain_percent"])
        for r in middle_rows
    )

    return {
        "mode": mode,
        "records": mode_records,
        "electrode_abs_strain_percent": electrode_abs_strain,
        "middle_abs_strain_percent": middle_abs_strain,
        "max_layer_abs_strain_percent": max(electrode_abs_strain, middle_abs_strain),
        "strain_imbalance_percent": abs(electrode_abs_strain - middle_abs_strain),
    }


def passes_mode_window(summary):
    """
    Filtering rule:
    - middle: only MoS2 must be 1.2–4.0%
    - electrodes: only Au electrodes must be 1.2–4.0%
    - shared: both Au and MoS2 must be 1.2–4.0%
    """
    mode = summary["mode"]
    au = summary["electrode_abs_strain_percent"]
    mos2 = summary["middle_abs_strain_percent"]

    if mode == "middle":
        return min_layer_strain_percent <= mos2 <= max_layer_strain_percent

    if mode == "electrodes":
        return min_layer_strain_percent <= au <= max_layer_strain_percent

    if mode == "shared":
        return (
            min_layer_strain_percent <= au <= max_layer_strain_percent
            and min_layer_strain_percent <= mos2 <= max_layer_strain_percent
        )

    raise ValueError(f"Unknown mode: {mode}")


def rank_key(row):
    """
    Smaller cells first. Then lower relevant strain.
    """
    mode = row["mode"]

    if mode == "middle":
        relevant = row["middle_abs_strain_percent"]
        imbalance = 0.0

    elif mode == "electrodes":
        relevant = row["electrode_abs_strain_percent"]
        imbalance = 0.0

    elif mode == "shared":
        relevant = row["max_layer_abs_strain_percent"]
        imbalance = row["strain_imbalance_percent"]

    else:
        relevant = row["max_layer_abs_strain_percent"]
        imbalance = row["strain_imbalance_percent"]

    return (
        row["sandwich_atoms"],
        relevant,
        imbalance,
        row["raw_match_max_strain_percent"],
    )


# -----------------------------
# Fast staged search
# -----------------------------
# Do not start with max_entry=6/max_area=60/search_limit=5000.
# These settings first target the old 41-atom-type cell family.

search_attempts = [
    dict(max_entry=3, max_area=6,  max_strain=0.080, max_atoms=60,  limit=100, search_limit=100),
    dict(max_entry=3, max_area=10, max_strain=0.080, max_atoms=80,  limit=150, search_limit=150),
    dict(max_entry=4, max_area=16, max_strain=0.090, max_atoms=120, limit=250, search_limit=250),
    dict(max_entry=4, max_area=25, max_strain=0.090, max_atoms=200, limit=400, search_limit=400),
]

wanted_modes = ["middle", "electrodes", "shared"]

mode_candidates = {mode: [] for mode in wanted_modes}
mode_objects = {mode: [] for mode in wanted_modes}
mode_records = {mode: [] for mode in wanted_modes}

used_settings = []

for settings in search_attempts:
    print("\nTrying search settings:")
    print(settings)

    t0 = time.time()

    trial_matches = find_2d_sandwich_matches(
        electrode=electrode,
        middle=middle,
        sort_by="atoms_then_strain",
        **settings,
    )

    elapsed = time.time() - t0
    print(f"Raw sandwich matches: {len(trial_matches)}")
    print(f"Elapsed seconds: {elapsed:.2f}")

    used_settings.append(settings)

    for sm in trial_matches:
        for mode in wanted_modes:
            summary = mode_strain_summary(sm, mode)
            if summary is None:
                continue

            if not passes_mode_window(summary):
                continue

            row = sm.as_dict()
            row.update(
                {
                    "mode": mode,
                    "shared_weight": shared_middle_weight if mode == "shared" else None,
                    "electrode_abs_strain_percent": summary["electrode_abs_strain_percent"],
                    "middle_abs_strain_percent": summary["middle_abs_strain_percent"],
                    "max_layer_abs_strain_percent": summary["max_layer_abs_strain_percent"],
                    "strain_imbalance_percent": summary["strain_imbalance_percent"],
                    "raw_match_max_strain_percent": 100.0 * sm.match.max_strain,
                }
            )

            mode_candidates[mode].append(row)
            mode_objects[mode].append(sm)
            mode_records[mode].append(summary["records"])

    for mode in wanted_modes:
        print(f"{mode} candidates so far: {len(mode_candidates[mode])}")

    # Stop once every mode has at least one candidate.
    if all(len(mode_candidates[mode]) > 0 for mode in wanted_modes):
        break


# -----------------------------
# Validate search result
# -----------------------------

missing = [mode for mode in wanted_modes if len(mode_candidates[mode]) == 0]

if missing:
    raise RuntimeError(
        "No valid small-cell candidates found for these modes: "
        f"{missing}. Try slightly widening the strain window or increasing max_area."
    )


# -----------------------------
# Rank candidates per mode
# -----------------------------

ranked_candidates = {}
ranked_objects = {}
ranked_records = {}

for mode in wanted_modes:
    ranked = sorted(
        zip(mode_candidates[mode], mode_objects[mode], mode_records[mode]),
        key=lambda x: rank_key(x[0]),
    )

    ranked_candidates[mode] = [x[0] for x in ranked]
    ranked_objects[mode] = [x[1] for x in ranked]
    ranked_records[mode] = [x[2] for x in ranked]

    print(f"\nBest {mode} candidate:")
    best = ranked_candidates[mode][0]
    print("sandwich atoms:", best["sandwich_atoms"])
    print(f"Au strain:   {best['electrode_abs_strain_percent']:.3f}%")
    print(f"MoS2 strain: {best['middle_abs_strain_percent']:.3f}%")
    print(f"raw match strain: {best['raw_match_max_strain_percent']:.3f}%")


# -----------------------------
# Save candidate tables
# -----------------------------

all_rows = []

for mode in wanted_modes:
    for rank, row in enumerate(ranked_candidates[mode]):
        row = dict(row)
        row["mode_rank"] = rank
        all_rows.append(row)

candidate_df = pd.DataFrame(all_rows)
display(candidate_df)

write_csv_records(outdir / "small_sandwich_candidates_all_modes.csv", all_rows)
write_json_records(outdir / "small_sandwich_candidates_all_modes.json", all_rows)


# -----------------------------
# Build and write structures
# -----------------------------

written_metadata = []

for mode in wanted_modes:
    for rank, sm in enumerate(ranked_objects[mode][:n_to_write_per_mode]):
        row = ranked_candidates[mode][rank]
        match = sm.match

        atoms = build_electrode_sandwich(
            electrode=electrode,
            middle=middle,
            match=match,
            electrode_middle_distance=electrode_middle_distance,
            vacuum=vacuum,
            strain=mode,
            shared_middle_weight=shared_middle_weight,
        )

        if len(atoms) > hard_max_atoms:
            raise RuntimeError(
                f"Built {mode} structure has {len(atoms)} atoms, exceeding {hard_max_atoms}."
            )

        stem = (
            f"{electrode_name}_{middle_name}_{electrode_name}"
            f"_{mode}_rank{rank:02d}"
            f"_{len(atoms)}atoms"
            f"_Au{row['electrode_abs_strain_percent']:.2f}pct"
            f"_MoS2{row['middle_abs_strain_percent']:.2f}pct"
        )

        write(outdir / f"{stem}.cif", atoms)
        write(outdir / f"{stem}.xyz", atoms, format="extxyz")
        write(outdir / f"{stem}.vasp", atoms, format="vasp", direct=True)

        records = ranked_records[mode][rank]

        write_csv_records(outdir / f"{stem}_layer_strain.csv", records)
        write_json_records(outdir / f"{stem}_layer_strain.json", records)

        written_metadata.append(
            {
                "mode": mode,
                "rank": rank,
                "stem": stem,
                "num_atoms": len(atoms),
                "candidate": row,
                "files": {
                    "cif": str(outdir / f"{stem}.cif"),
                    "xyz": str(outdir / f"{stem}.xyz"),
                    "vasp": str(outdir / f"{stem}.vasp"),
                    "strain_csv": str(outdir / f"{stem}_layer_strain.csv"),
                },
            }
        )

        print(f"Wrote {stem}")


with open(outdir / "written_small_sandwich_all_modes_metadata.json", "w") as f:
    json.dump(
        {
            "strain_window_percent": [
                min_layer_strain_percent,
                max_layer_strain_percent,
            ],
            "hard_max_atoms": hard_max_atoms,
            "used_settings": used_settings,
            "written": written_metadata,
        },
        f,
        indent=2,
    )


print("\nOutput directory:", outdir.resolve())
print("Candidate table:")
print(outdir / "small_sandwich_candidates_all_modes.csv")
print("Metadata:")
print(outdir / "written_small_sandwich_all_modes_metadata.json")

Electrode atoms in input cell: 4
Middle atoms in input cell: 3
Electrode cell:
Cell([[2.883581453678741, 0.0, 0.0], [1.4417907268393704, 2.4972547927674507, 0.0], [0.0, 0.0, 37.06330319326588]])
Middle cell:
Cell([[3.18, 0.0, 0.0], [-1.5900000000000007, 2.7539607840345144, 0.0], [0.0, 0.0, 33.19]])

Trying search settings:
{'max_entry': 3, 'max_area': 6, 'max_strain': 0.08, 'max_atoms': 60, 'limit': 100, 'search_limit': 100}
Raw sandwich matches: 100
Elapsed seconds: 26.67
middle candidates so far: 0
electrodes candidates so far: 0
shared candidates so far: 0

Trying search settings:
{'max_entry': 3, 'max_area': 10, 'max_strain': 0.08, 'max_atoms': 80, 'limit': 150, 'search_limit': 150}
Raw sandwich matches: 6
Elapsed seconds: 41.74
middle candidates so far: 0
electrodes candidates so far: 0
shared candidates so far: 0

Trying search settings:
{'max_entry': 4, 'max_area': 16, 'max_strain': 0.09, 'max_atoms': 120, 'limit': 250, 'search_limit': 250}
Raw sandwich matches: 106
Elapsed seco

RuntimeError: No valid small-cell candidates found for these modes: ['shared']. Try slightly widening the strain window or increasing max_area.

In [6]:
import numpy as np
import pandas as pd
from pathlib import Path
from ase.io import read, write

from matstruct_lab.lattice_match import (
    Match2D,
    inplane_cell,
    deformation_strain,
    det_int,
)
from matstruct_lab.heterostructures import build_electrode_sandwich
from matstruct_lab.sandwich_match import sandwich_layer_strain_records


electrode = read("../examples/structures/Au_111.cif")
middle = read("../examples/structures/MoS2.cif")

electrode_name = "Au111"
middle_name = "MoS2"

outdir = Path("forced_41_atom_check")
outdir.mkdir(exist_ok=True)

# Known small-family matrices:
# electrode determinant = 4
# MoS2 determinant = 3
# sandwich atoms = 2*(4 Au atoms*4) + (3 MoS2 atoms*3) = 41
Mb = np.array([[-2, 2],
               [ 0,-2]], dtype=int)

Mt = np.array([[-1, 1],
               [-1,-2]], dtype=int)

bottom_cell = Mb @ inplane_cell(electrode)
top_cell = Mt @ inplane_cell(middle)

max_s, rms_s, area_s = deformation_strain(top_cell, bottom_cell)

forced_match = Match2D(
    bottom_matrix=Mb,
    top_matrix=Mt,
    bottom_area_multiplier=det_int(Mb),
    top_area_multiplier=det_int(Mt),
    max_strain=max_s,
    rms_strain=rms_s,
    area_strain=area_s,
    total_atoms=len(electrode) * det_int(Mb) + len(middle) * det_int(Mt),
)

sandwich_atoms = (
    2 * len(electrode) * forced_match.bottom_area_multiplier
    + len(middle) * forced_match.top_area_multiplier
)

print("Forced small-cell candidate")
print("bilayer proxy atoms:", forced_match.total_atoms)
print("sandwich atoms:", sandwich_atoms)
print(f"raw top->bottom max strain: {100*forced_match.max_strain:.3f}%")
print(f"raw top->bottom rms strain: {100*forced_match.rms_strain:.3f}%")
print(f"raw area strain: {100*forced_match.area_strain:.3f}%")
print("electrode matrix:")
print(forced_match.bottom_matrix)
print("middle matrix:")
print(forced_match.top_matrix)


for mode in ["middle", "electrodes", "shared"]:
    atoms = build_electrode_sandwich(
        electrode=electrode,
        middle=middle,
        match=forced_match,
        electrode_middle_distance=3.3,
        vacuum=20.0,
        strain=mode,
        shared_middle_weight=0.5,
    )

    stem = f"{electrode_name}_{middle_name}_{electrode_name}_forced41_{mode}_{len(atoms)}atoms"

    write(outdir / f"{stem}.vasp", atoms, format="vasp", direct=True)
    write(outdir / f"{stem}.cif", atoms)
    write(outdir / f"{stem}.xyz", atoms, format="extxyz")

    records = sandwich_layer_strain_records(
        electrode=electrode,
        middle=middle,
        sandwich_match=forced_match,
        electrode_name=electrode_name,
        middle_name=middle_name,
        shared_middle_weight=0.5,
    )

    mode_records = [r for r in records if r["strain_mode"] == mode]
    df = pd.DataFrame(mode_records)

    print("\nMode:", mode)
    display(df[[
        "structure",
        "layer",
        "strained",
        "max_abs_principal_strain_percent",
        "rms_principal_strain_percent",
        "signed_area_strain_percent",
    ]])

    df.to_csv(outdir / f"{stem}_layer_strain.csv", index=False)

print("Wrote forced 41-atom checks to:", outdir.resolve())

Forced small-cell candidate
bilayer proxy atoms: 25
sandwich atoms: 41
raw top->bottom max strain: 4.707%
raw top->bottom rms strain: 4.707%
raw area strain: 9.635%
electrode matrix:
[[-2  2]
 [ 0 -2]]
middle matrix:
[[-1  1]
 [-1 -2]]

Mode: middle


,structure,layer,strained,max_abs_principal_strain_percent,rms_principal_strain_percent,signed_area_strain_percent
0,Au111_MoS2_Au111_middle_strained,bottom_Au111,False,1.110223e-14,7.850462e-15,0.00000
1,Au111_MoS2_Au111_middle_strained,MoS2,True,4.706700e+00,4.706700e+00,9.63493
2,Au111_MoS2_Au111_middle_strained,top_Au111,False,1.110223e-14,7.850462e-15,0.00000



Mode: electrodes


,structure,layer,strained,max_abs_principal_strain_percent,rms_principal_strain_percent,signed_area_strain_percent
0,Au111_MoS2_Au111_electrodes_strained,bottom_Au111,True,4.495128e+00,4.495128e+00,-8.788194
1,Au111_MoS2_Au111_electrodes_strained,MoS2,False,1.110223e-14,7.850462e-15,0.000000
2,Au111_MoS2_Au111_electrodes_strained,top_Au111,True,4.495128e+00,4.495128e+00,-8.788194



Mode: shared


,structure,layer,strained,max_abs_principal_strain_percent,rms_principal_strain_percent,signed_area_strain_percent
0,Au111_MoS2_Au111_shared_strain,bottom_Au111,True,5.576605,5.576605,-10.842226
1,Au111_MoS2_Au111_shared_strain,MoS2,True,1.132380,1.132380,-2.251936
2,Au111_MoS2_Au111_shared_strain,top_Au111,True,5.576605,5.576605,-10.842226


Wrote forced 41-atom checks to: /home/brian/Research/matstruct-lab/notebooks/forced_41_atom_check


In [7]:
import numpy as np
import pandas as pd
from pathlib import Path
from ase.io import read, write
from ase import Atoms

from matstruct_lab.lattice_match import (
    Match2D,
    inplane_cell,
    deformation_strain,
    det_int,
    make_2d_supercell,
)
from matstruct_lab.heterostructures import set_inplane_cell, shift_z_min_to


# -----------------------------
# Inputs
# -----------------------------

electrode = read("../examples/structures/Au_111.cif")
middle = read("../examples/structures/MoS2.cif")

electrode_name = "Au111"
middle_name = "MoS2"

outdir = Path("balanced_41_atom_shared")
outdir.mkdir(exist_ok=True)

electrode_middle_distance = 3.3
vacuum = 20.0

# Old-style shared interpolation.
# alpha = 0.0 -> MoS2-like cell / Au strained
# alpha = 1.0 -> Au-like cell / MoS2 strained
# alpha = 0.5 -> balanced shared strain
alpha = 0.5


# -----------------------------
# Forced known small-cell matrices
# -----------------------------

Mb = np.array([[-2,  2],
               [ 0, -2]], dtype=int)

Mt = np.array([[-1,  1],
               [-1, -2]], dtype=int)

bottom_cell = Mb @ inplane_cell(electrode)
top_cell = Mt @ inplane_cell(middle)

max_s, rms_s, area_s = deformation_strain(top_cell, bottom_cell)

match = Match2D(
    bottom_matrix=Mb,
    top_matrix=Mt,
    bottom_area_multiplier=det_int(Mb),
    top_area_multiplier=det_int(Mt),
    max_strain=max_s,
    rms_strain=rms_s,
    area_strain=area_s,
    total_atoms=len(electrode) * det_int(Mb) + len(middle) * det_int(Mt),
)

sandwich_atoms = (
    2 * len(electrode) * match.bottom_area_multiplier
    + len(middle) * match.top_area_multiplier
)

print("Sandwich atoms:", sandwich_atoms)
print(f"Raw MoS2 -> Au mismatch: {100 * match.max_strain:.3f}%")


# -----------------------------
# Balanced geometric shared cell
# -----------------------------

def balanced_hex_shared_cell(electrode_cell, middle_cell, alpha=0.5):
    """
    Build a balanced shared cell using geometric interpolation of the
    rotation-invariant lattice mismatch.

    Assumes this small Au/MoS2 case is close to isotropic hexagonal mismatch.
    """
    F = np.linalg.solve(middle_cell, electrode_cell)
    singular_values = np.linalg.svd(F, compute_uv=False)

    if np.ptp(singular_values) > 1e-4:
        print("Warning: mismatch is not perfectly isotropic.")
        print("singular values:", singular_values)

    s = float(np.mean(singular_values))

    # alpha = 1 -> electrode cell
    # alpha = 0 -> middle-length cell, expressed in electrode orientation
    scale_from_electrode = s ** (alpha - 1.0)

    return electrode_cell * scale_from_electrode


target_inplane = balanced_hex_shared_cell(
    electrode_cell=bottom_cell,
    middle_cell=top_cell,
    alpha=alpha,
)

print("Target in-plane cell:")
print(target_inplane)


# -----------------------------
# Strain reporting
# -----------------------------

def strain_report(reference_cell, target_cell, reference, target):
    F = np.linalg.solve(reference_cell, target_cell)
    principal = np.linalg.svd(F, compute_uv=False) - 1.0

    area_strain = np.linalg.det(F) - 1.0

    return {
        "reference": reference,
        "target": target,
        "principal_strain_1_percent": 100 * float(principal[0]),
        "principal_strain_2_percent": 100 * float(principal[1]),
        "max_abs_principal_strain_percent": 100 * float(np.max(np.abs(principal))),
        "rms_principal_strain_percent": 100 * float(np.sqrt(np.mean(principal**2))),
        "signed_area_strain_percent": 100 * float(area_strain),
    }


strain_rows = [
    strain_report(bottom_cell, target_inplane, "Au matched supercell", "balanced shared cell"),
    strain_report(top_cell, target_inplane, "MoS2 matched supercell", "balanced shared cell"),
]

strain_df = pd.DataFrame(strain_rows)
display(strain_df)

strain_df.to_csv(outdir / "balanced_41_atom_shared_strain.csv", index=False)


# -----------------------------
# Build sandwich manually
# -----------------------------

bottom_electrode = make_2d_supercell(electrode, match.bottom_matrix)
middle_sc = make_2d_supercell(middle, match.top_matrix)
top_electrode = make_2d_supercell(electrode, match.bottom_matrix)

bottom_electrode = set_inplane_cell(bottom_electrode, target_inplane, scale_atoms=True)
middle_sc = set_inplane_cell(middle_sc, target_inplane, scale_atoms=True)
top_electrode = set_inplane_cell(top_electrode, target_inplane, scale_atoms=True)

bottom_electrode = shift_z_min_to(bottom_electrode, vacuum / 2.0)

middle_sc = shift_z_min_to(
    middle_sc,
    bottom_electrode.positions[:, 2].max() + electrode_middle_distance,
)

top_electrode = shift_z_min_to(
    top_electrode,
    middle_sc.positions[:, 2].max() + electrode_middle_distance,
)

stack = bottom_electrode + middle_sc + top_electrode

z_min = stack.positions[:, 2].min()
z_max = stack.positions[:, 2].max()
final_z = (z_max - z_min) + vacuum

final_cell = np.zeros((3, 3), dtype=float)
final_cell[:2, :2] = target_inplane
final_cell[2, 2] = final_z

stack.set_cell(final_cell, scale_atoms=False)
stack.set_pbc([True, True, False])
stack.wrap(eps=1e-8)

print("Built atoms:", len(stack))

stem = (
    f"{electrode_name}_{middle_name}_{electrode_name}"
    f"_balanced_shared_alpha{alpha:.2f}_{len(stack)}atoms"
)

write(outdir / f"{stem}.vasp", stack, format="vasp", direct=True)
write(outdir / f"{stem}.cif", stack)
write(outdir / f"{stem}.xyz", stack, format="extxyz")

print("Wrote:")
print(outdir / f"{stem}.vasp")
print(outdir / f"{stem}.cif")
print(outdir / f"{stem}.xyz")
print(outdir / "balanced_41_atom_shared_strain.csv")

Sandwich atoms: 41
Raw MoS2 -> Au mismatch: 4.707%
Target in-plane cell:
[[-2.81802595  4.88096412]
 [-2.81802595 -4.88096412]]


,reference,target,principal_strain_1_percent,principal_strain_2_percent,max_abs_principal_strain_percent,rms_principal_strain_percent,signed_area_strain_percent
0,Au matched supercell,balanced shared cell,-2.273406,-2.273406,2.273406,2.273406,-4.495128
1,MoS2 matched supercell,balanced shared cell,2.326292,2.326292,2.326292,2.326292,4.706700


Built atoms: 41
Wrote:
balanced_41_atom_shared/Au111_MoS2_Au111_balanced_shared_alpha0.50_41atoms.vasp
balanced_41_atom_shared/Au111_MoS2_Au111_balanced_shared_alpha0.50_41atoms.cif
balanced_41_atom_shared/Au111_MoS2_Au111_balanced_shared_alpha0.50_41atoms.xyz
balanced_41_atom_shared/balanced_41_atom_shared_strain.csv


In [8]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

from ase import Atoms
from ase.io import read, write
from ase.build import fcc111

from matstruct_lab.lattice_match import make_2d_supercell, inplane_cell
from matstruct_lab.heterostructures import set_inplane_cell, shift_z_min_to


# ============================================================
# User settings
# ============================================================

structure_dir = Path("../examples/structures")
mos2_path = structure_dir / "MoS2.cif"

outdir = Path("metal_mos2_metal_alpha_sweep")
outdir.mkdir(parents=True, exist_ok=True)

metals = ["Au", "Ag", "Cu", "Pt", "Pd", "Ni"]

alphas = [0.0, 0.5, 1.0]

# 2.5–4.0 Å inclusive.
interlayer_distances = np.arange(2.5, 4.0001, 0.25)

vacuum = 20.0

# 41-atom family for 4-layer fcc(111) metal input cells:
# 2 * (4 metal atoms * det=4) + (3 MoS2 atoms * det=3) = 41
metal_matrix = np.array([[-2,  2],
                         [ 0, -2]], dtype=int)

mos2_matrix = np.array([[-1,  1],
                        [-1, -2]], dtype=int)

# Used only if a metal_111 structure file is not found.
# These are typical room-temperature fcc cubic lattice constants in Å.
# Prefer your DFT-relaxed structures if you have them.
fcc_lattice_constants_A = {
    "Au": 4.0782,
    "Ag": 4.0862,
    "Cu": 3.6149,
    "Pt": 3.9239,
    "Pd": 3.8907,
    "Ni": 3.5240,
}

fcc111_layers = 4

# Generate extra structures for non-Au metals forced to Au-derived alpha cells.
generate_au_matched_series = True


# ============================================================
# Geometry and strain helpers
# ============================================================

def find_structure_file_for_metal(metal: str) -> Path | None:
    candidates = [
        structure_dir / f"{metal}_111.cif",
        structure_dir / f"{metal}_111.vasp",
        structure_dir / f"{metal}_111.POSCAR",
        structure_dir / f"{metal}111.cif",
        structure_dir / f"{metal}111.vasp",
        structure_dir / f"{metal}.cif",
    ]

    for path in candidates:
        if path.exists():
            return path

    return None


def load_or_build_fcc111(metal: str) -> tuple[Atoms, str]:
    path = find_structure_file_for_metal(metal)

    if path is not None:
        atoms = read(path)
        atoms.set_pbc([True, True, False])
        return atoms, f"file:{path}"

    if metal not in fcc_lattice_constants_A:
        raise ValueError(f"No lattice constant provided for {metal}")

    atoms = fcc111(
        metal,
        size=(1, 1, fcc111_layers),
        a=fcc_lattice_constants_A[metal],
        vacuum=10.0,
        orthogonal=False,
    )
    atoms.set_pbc([True, True, False])

    return atoms, f"generated:fcc111(a={fcc_lattice_constants_A[metal]}, layers={fcc111_layers})"


def check_2d_cell(atoms: Atoms, name: str, tol: float = 1e-6) -> None:
    cell = atoms.cell.array

    if abs(cell[0, 2]) > tol or abs(cell[1, 2]) > tol:
        raise ValueError(
            f"{name}: first two lattice vectors have nonzero z components. "
            "Reorient the slab so a,b are in-plane and c is out-of-plane."
        )

    if abs(cell[2, 0]) > tol or abs(cell[2, 1]) > tol:
        print(
            f"Warning: {name}: c vector has x/y components. "
            "An orthogonal vacuum direction is safer."
        )


def cell_metrics(cell2d: np.ndarray) -> dict:
    cell2d = np.asarray(cell2d, dtype=float)

    a = cell2d[0]
    b = cell2d[1]

    a_len = float(np.linalg.norm(a))
    b_len = float(np.linalg.norm(b))

    if a_len == 0 or b_len == 0:
        raise ValueError("Zero-length in-plane vector.")

    cos_gamma = float(np.dot(a, b) / (a_len * b_len))
    cos_gamma = float(np.clip(cos_gamma, -1.0, 1.0))

    gamma = float(np.degrees(np.arccos(cos_gamma)))
    area = float(abs(np.linalg.det(cell2d)))

    return {
        "a_A": a_len,
        "b_A": b_len,
        "gamma_deg": gamma,
        "area_A2": area,
    }


def add_cell_record(prefix: str, cell2d: np.ndarray) -> dict:
    m = cell_metrics(cell2d)

    return {
        f"{prefix}_a_x": float(cell2d[0, 0]),
        f"{prefix}_a_y": float(cell2d[0, 1]),
        f"{prefix}_b_x": float(cell2d[1, 0]),
        f"{prefix}_b_y": float(cell2d[1, 1]),
        f"{prefix}_a_A": m["a_A"],
        f"{prefix}_b_A": m["b_A"],
        f"{prefix}_gamma_deg": m["gamma_deg"],
        f"{prefix}_area_A2": m["area_A2"],
    }


def strain_record(reference_cell: np.ndarray, target_cell: np.ndarray, prefix: str) -> dict:
    """
    Rotation-invariant 2D principal strain from reference_cell -> target_cell.

    Positive principal strain = tensile/stretch.
    Negative principal strain = compressive.
    """
    reference_cell = np.asarray(reference_cell, dtype=float)
    target_cell = np.asarray(target_cell, dtype=float)

    F = np.linalg.solve(reference_cell, target_cell)
    singular_values = np.linalg.svd(F, compute_uv=False)
    principal = singular_values - 1.0

    ref_m = cell_metrics(reference_cell)
    tar_m = cell_metrics(target_cell)

    signed_area_strain = (tar_m["area_A2"] - ref_m["area_A2"]) / ref_m["area_A2"]

    return {
        f"{prefix}_principal_strain_1_percent": 100.0 * float(principal[0]),
        f"{prefix}_principal_strain_2_percent": 100.0 * float(principal[1]),
        f"{prefix}_max_abs_principal_strain_percent": 100.0 * float(np.max(np.abs(principal))),
        f"{prefix}_rms_principal_strain_percent": 100.0 * float(np.sqrt(np.mean(principal**2))),
        f"{prefix}_signed_area_strain_percent": 100.0 * float(signed_area_strain),
        f"{prefix}_deformation_gradient": F.tolist(),
    }


def raw_mismatch_record(metal_cell: np.ndarray, mos2_cell: np.ndarray) -> dict:
    """
    Raw mismatch from MoS2 matched cell to metal matched cell.
    """
    rec = strain_record(mos2_cell, metal_cell, "raw_mos2_to_metal")
    return rec


def alpha_mode_label(alpha: float, series: str) -> str:
    if series == "native_alpha":
        if np.isclose(alpha, 0.0):
            return "MoS2-fixed-length / metal strained"
        if np.isclose(alpha, 0.5):
            return "balanced shared biaxial strain"
        if np.isclose(alpha, 1.0):
            return "metal-fixed-length / MoS2 strained"
        return "interpolated native alpha"

    if series == "au_matched_alpha":
        return "forced to Au-derived alpha target cell"

    return "unknown"


def balanced_alpha_target_cell(
    metal_cell: np.ndarray,
    mos2_cell: np.ndarray,
    alpha: float,
) -> tuple[np.ndarray, dict]:
    """
    Return an alpha-interpolated target cell in the metal-cell orientation.

    alpha = 0.0 gives a MoS2-like length scale in the metal orientation.
    alpha = 0.5 gives a balanced geometric shared cell.
    alpha = 1.0 gives the native metal matched cell.

    This avoids direct matrix averaging, which can fail for rotated/sign-equivalent
    supercell matrices.
    """
    metal_cell = np.asarray(metal_cell, dtype=float)
    mos2_cell = np.asarray(mos2_cell, dtype=float)

    F = np.linalg.solve(mos2_cell, metal_cell)
    singular_values = np.linalg.svd(F, compute_uv=False)

    # Geometric mean gives an area-consistent isotropic scale.
    scale_mos2_to_metal = float(np.exp(np.mean(np.log(singular_values))))

    # alpha=1 -> metal_cell
    # alpha=0 -> metal_cell / scale_mos2_to_metal
    target = metal_cell * (scale_mos2_to_metal ** (alpha - 1.0))

    info = {
        "alpha_scale_mos2_to_metal": scale_mos2_to_metal,
        "alpha_singular_value_1": float(singular_values[0]),
        "alpha_singular_value_2": float(singular_values[1]),
        "alpha_singular_value_spread": float(abs(singular_values[0] - singular_values[1])),
    }

    return target, info


def build_manual_sandwich(
    metal_atoms: Atoms,
    mos2_atoms: Atoms,
    target_inplane: np.ndarray,
    interlayer_distance: float,
) -> Atoms:
    """
    Build metal / MoS2 / metal sandwich using a specified target in-plane cell.
    """
    bottom_metal = make_2d_supercell(metal_atoms, metal_matrix)
    middle_sc = make_2d_supercell(mos2_atoms, mos2_matrix)
    top_metal = make_2d_supercell(metal_atoms, metal_matrix)

    bottom_metal = set_inplane_cell(bottom_metal, target_inplane, scale_atoms=True)
    middle_sc = set_inplane_cell(middle_sc, target_inplane, scale_atoms=True)
    top_metal = set_inplane_cell(top_metal, target_inplane, scale_atoms=True)

    bottom_metal = shift_z_min_to(bottom_metal, vacuum / 2.0)

    middle_sc = shift_z_min_to(
        middle_sc,
        bottom_metal.positions[:, 2].max() + interlayer_distance,
    )

    top_metal = shift_z_min_to(
        top_metal,
        middle_sc.positions[:, 2].max() + interlayer_distance,
    )

    stack = bottom_metal + middle_sc + top_metal

    z_min = stack.positions[:, 2].min()
    z_max = stack.positions[:, 2].max()
    final_z = (z_max - z_min) + vacuum

    final_cell = np.zeros((3, 3), dtype=float)
    final_cell[:2, :2] = target_inplane
    final_cell[2, 2] = final_z

    stack.set_cell(final_cell, scale_atoms=False)
    stack.set_pbc([True, True, False])
    stack.wrap(eps=1e-8)

    return stack


def safe_float_token(x: float) -> str:
    return f"{x:.2f}".replace(".", "p")


def write_all_formats(atoms: Atoms, stem: str) -> dict:
    paths = {
        "vasp": outdir / f"{stem}.vasp",
        "cif": outdir / f"{stem}.cif",
        "xyz": outdir / f"{stem}.xyz",
    }

    write(paths["vasp"], atoms, format="vasp", direct=True)
    write(paths["cif"], atoms)
    write(paths["xyz"], atoms, format="extxyz")

    return {k: str(v) for k, v in paths.items()}


# ============================================================
# Load structures
# ============================================================

if not mos2_path.exists():
    raise FileNotFoundError(f"Could not find MoS2 structure: {mos2_path}")

mos2 = read(mos2_path)
mos2.set_pbc([True, True, False])
check_2d_cell(mos2, "MoS2")

metal_atoms_by_name = {}
metal_source_by_name = {}

for metal in metals:
    atoms, source = load_or_build_fcc111(metal)
    check_2d_cell(atoms, metal)
    metal_atoms_by_name[metal] = atoms
    metal_source_by_name[metal] = source

    print(f"{metal}: {len(atoms)} atoms in input cell | {source}")

print("MoS2:", len(mos2), "atoms in input cell |", mos2_path)


# ============================================================
# Precompute matched reference cells
# ============================================================

mos2_sc = make_2d_supercell(mos2, mos2_matrix)
mos2_ref_cell = inplane_cell(mos2_sc)

metal_ref_cell_by_name = {}

for metal, atoms in metal_atoms_by_name.items():
    metal_sc = make_2d_supercell(atoms, metal_matrix)
    metal_ref_cell_by_name[metal] = inplane_cell(metal_sc)

# Au-derived target cells for comparison series.
au_target_by_alpha = {}
au_alpha_info_by_alpha = {}

for alpha in alphas:
    target_cell, alpha_info = balanced_alpha_target_cell(
        metal_cell=metal_ref_cell_by_name["Au"],
        mos2_cell=mos2_ref_cell,
        alpha=alpha,
    )
    au_target_by_alpha[alpha] = target_cell
    au_alpha_info_by_alpha[alpha] = alpha_info


# ============================================================
# Generate structures
# ============================================================

records = []
summary_records = []

for metal in metals:
    metal_atoms = metal_atoms_by_name[metal]
    metal_ref_cell = metal_ref_cell_by_name[metal]

    series_jobs = []

    # Native alpha series for each metal.
    for alpha in alphas:
        target_cell, alpha_info = balanced_alpha_target_cell(
            metal_cell=metal_ref_cell,
            mos2_cell=mos2_ref_cell,
            alpha=alpha,
        )

        series_jobs.append(
            {
                "series": "native_alpha",
                "metal": metal,
                "alpha": alpha,
                "target_cell": target_cell,
                "alpha_info": alpha_info,
                "mode_label": alpha_mode_label(alpha, "native_alpha"),
                "forced_reference": None,
            }
        )

    # Additional forced-to-Au alpha target series for non-Au metals.
    if generate_au_matched_series and metal != "Au":
        for alpha in alphas:
            series_jobs.append(
                {
                    "series": "au_matched_alpha",
                    "metal": metal,
                    "alpha": alpha,
                    "target_cell": au_target_by_alpha[alpha],
                    "alpha_info": au_alpha_info_by_alpha[alpha],
                    "mode_label": alpha_mode_label(alpha, "au_matched_alpha"),
                    "forced_reference": f"Au_native_alpha_{alpha}",
                }
            )

    for job in series_jobs:
        series = job["series"]
        alpha = job["alpha"]
        target_cell = job["target_cell"]

        metal_strain = strain_record(metal_ref_cell, target_cell, "metal")
        mos2_strain = strain_record(mos2_ref_cell, target_cell, "mos2")
        raw_strain = raw_mismatch_record(metal_ref_cell, mos2_ref_cell)

        base_record = {
            "series": series,
            "metal": metal,
            "middle": "MoS2",
            "structure_type": f"{metal}/MoS2/{metal}",
            "alpha": alpha,
            "mode_label": job["mode_label"],
            "forced_reference": job["forced_reference"],
            "metal_source": metal_source_by_name[metal],
            "mos2_source": str(mos2_path),
            "metal_matrix": metal_matrix.tolist(),
            "mos2_matrix": mos2_matrix.tolist(),
            "metal_input_atoms": len(metal_atoms),
            "mos2_input_atoms": len(mos2),
            "metal_supercell_atoms_one_electrode": len(metal_atoms) * abs(round(np.linalg.det(metal_matrix))),
            "mos2_supercell_atoms": len(mos2) * abs(round(np.linalg.det(mos2_matrix))),
            "sandwich_atoms_expected": (
                2 * len(metal_atoms) * abs(round(np.linalg.det(metal_matrix)))
                + len(mos2) * abs(round(np.linalg.det(mos2_matrix)))
            ),
            **job["alpha_info"],
            **add_cell_record("metal_reference", metal_ref_cell),
            **add_cell_record("mos2_reference", mos2_ref_cell),
            **add_cell_record("target", target_cell),
            **raw_strain,
            **metal_strain,
            **mos2_strain,
        }

        summary_records.append(dict(base_record))

        for distance in interlayer_distances:
            atoms = build_manual_sandwich(
                metal_atoms=metal_atoms,
                mos2_atoms=mos2,
                target_inplane=target_cell,
                interlayer_distance=float(distance),
            )

            stem = (
                f"{metal}_MoS2_{metal}"
                f"_{series}"
                f"_alpha{safe_float_token(alpha)}"
                f"_d{safe_float_token(float(distance))}A"
                f"_{len(atoms)}atoms"
            )

            files = write_all_formats(atoms, stem)

            rec = dict(base_record)
            rec.update(
                {
                    "interlayer_distance_A": float(distance),
                    "actual_atoms": len(atoms),
                    "stem": stem,
                    "vasp": files["vasp"],
                    "cif": files["cif"],
                    "xyz": files["xyz"],
                }
            )

            records.append(rec)

            print(
                f"Wrote {stem} | "
                f"metal strain={rec['metal_max_abs_principal_strain_percent']:.3f}% | "
                f"MoS2 strain={rec['mos2_max_abs_principal_strain_percent']:.3f}%"
            )


# ============================================================
# Save metadata tables
# ============================================================

metadata_df = pd.DataFrame(records)
summary_df = pd.DataFrame(summary_records)

metadata_csv = outdir / "all_generated_structures_metadata.csv"
summary_csv = outdir / "unique_alpha_targets_strain_summary.csv"

metadata_json = outdir / "all_generated_structures_metadata.json"
summary_json = outdir / "unique_alpha_targets_strain_summary.json"

metadata_df.to_csv(metadata_csv, index=False)
summary_df.to_csv(summary_csv, index=False)

metadata_df.to_json(metadata_json, orient="records", indent=2)
summary_df.to_json(summary_json, orient="records", indent=2)

print("\nSaved:")
print(metadata_csv)
print(summary_csv)
print(metadata_json)
print(summary_json)

display_cols = [
    "series",
    "metal",
    "alpha",
    "mode_label",
    "forced_reference",
    "sandwich_atoms_expected",
    "target_a_A",
    "target_b_A",
    "target_gamma_deg",
    "target_area_A2",
    "metal_max_abs_principal_strain_percent",
    "metal_principal_strain_1_percent",
    "metal_principal_strain_2_percent",
    "mos2_max_abs_principal_strain_percent",
    "mos2_principal_strain_1_percent",
    "mos2_principal_strain_2_percent",
    "raw_mos2_to_metal_max_abs_principal_strain_percent",
]

display(summary_df[display_cols].sort_values(["series", "metal", "alpha"]))

Au: 4 atoms in input cell | file:../examples/structures/Au_111.cif
Ag: 4 atoms in input cell | generated:fcc111(a=4.0862, layers=4)
Cu: 4 atoms in input cell | generated:fcc111(a=3.6149, layers=4)
Pt: 4 atoms in input cell | generated:fcc111(a=3.9239, layers=4)
Pd: 4 atoms in input cell | generated:fcc111(a=3.8907, layers=4)
Ni: 4 atoms in input cell | generated:fcc111(a=3.524, layers=4)
MoS2: 3 atoms in input cell | ../examples/structures/MoS2.cif
Wrote Au_MoS2_Au_native_alpha_alpha0p00_d2p50A_41atoms | metal strain=4.495% | MoS2 strain=0.000%
Wrote Au_MoS2_Au_native_alpha_alpha0p00_d2p75A_41atoms | metal strain=4.495% | MoS2 strain=0.000%
Wrote Au_MoS2_Au_native_alpha_alpha0p00_d3p00A_41atoms | metal strain=4.495% | MoS2 strain=0.000%
Wrote Au_MoS2_Au_native_alpha_alpha0p00_d3p25A_41atoms | metal strain=4.495% | MoS2 strain=0.000%
Wrote Au_MoS2_Au_native_alpha_alpha0p00_d3p50A_41atoms | metal strain=4.495% | MoS2 strain=0.000%
Wrote Au_MoS2_Au_native_alpha_alpha0p00_d3p75A_41atoms | 

,series,metal,alpha,mode_label,forced_reference,sandwich_atoms_expected,target_a_A,target_b_A,target_gamma_deg,target_area_A2,metal_max_abs_principal_strain_percent,metal_principal_strain_1_percent,metal_principal_strain_2_percent,mos2_max_abs_principal_strain_percent,mos2_principal_strain_1_percent,mos2_principal_strain_2_percent,raw_mos2_to_metal_max_abs_principal_strain_percent
6,au_matched_alpha,Ag,0.0,forced to Au-derived alpha target cell,Au_native_alpha_0.0,41,5.507922,5.507922,120.0,26.272786,4.686783e+00,-4.686783e+00,-4.686783e+00,4.440892e-14,4.440892e-14,0.000000e+00,4.917243
7,au_matched_alpha,Ag,0.5,forced to Au-derived alpha target cell,Au_native_alpha_0.5,41,5.636052,5.636052,120.0,27.509367,2.469519e+00,-2.469519e+00,-2.469519e+00,2.326292e+00,2.326292e+00,2.326292e+00,4.917243
8,au_matched_alpha,Ag,1.0,forced to Au-derived alpha target cell,Au_native_alpha_1.0,41,5.767163,5.767163,120.0,28.804150,2.006754e-01,-2.006754e-01,-2.006754e-01,4.706700e+00,4.706700e+00,4.706700e+00,4.917243
12,au_matched_alpha,Cu,0.0,forced to Au-derived alpha target cell,Au_native_alpha_0.0,41,5.507922,5.507922,120.0,26.272786,7.739874e+00,7.739874e+00,7.739874e+00,4.440892e-14,4.440892e-14,0.000000e+00,7.183853
13,au_matched_alpha,Cu,0.5,forced to Au-derived alpha target cell,Au_native_alpha_0.5,41,5.636052,5.636052,120.0,27.509367,1.024622e+01,1.024622e+01,1.024622e+01,2.326292e+00,2.326292e+00,2.326292e+00,7.183853
14,au_matched_alpha,Cu,1.0,forced to Au-derived alpha target cell,Au_native_alpha_1.0,41,5.767163,5.767163,120.0,28.804150,1.281087e+01,1.281087e+01,1.281087e+01,4.706700e+00,4.706700e+00,4.706700e+00,7.183853
30,au_matched_alpha,Ni,0.0,forced to Au-derived alpha target cell,Au_native_alpha_0.0,41,5.507922,5.507922,120.0,26.272786,1.051898e+01,1.051898e+01,1.051898e+01,4.440892e-14,4.440892e-14,0.000000e+00,9.517800
31,au_matched_alpha,Ni,0.5,forced to Au-derived alpha target cell,Au_native_alpha_0.5,41,5.636052,5.636052,120.0,27.509367,1.308997e+01,1.308997e+01,1.308997e+01,2.326292e+00,2.326292e+00,2.326292e+00,9.517800
32,au_matched_alpha,Ni,1.0,forced to Au-derived alpha target cell,Au_native_alpha_1.0,41,5.767163,5.767163,120.0,28.804150,1.572077e+01,1.572077e+01,1.572077e+01,4.706700e+00,4.706700e+00,4.706700e+00,9.517800
24,au_matched_alpha,Pd,0.0,forced to Au-derived alpha target cell,Au_native_alpha_0.0,41,5.507922,5.507922,120.0,26.272786,1.025186e-01,1.025186e-01,1.025186e-01,4.440892e-14,4.440892e-14,0.000000e+00,0.102414
